# IoT Sensor Data Platform: Governance, Quality & Predictive Analytics
**Author:** _your name_ | **Tools:** Python, pandas, SQL (SQLite), scikit-learn, Tableau Public

**Business context:** An IoT company (like Kreon, with wireless and wired sensor products) collects telemetry from devices in the field.
This project builds a small, governed data platform: ingest raw telemetry, validate and clean it, document it, analyse it, detect failing devices and forecast battery replacement.

**How to run:** Runtime > Run all. Takes about 1 minute. Download the zip at the end.

Layers: `raw_telemetry` (never modified) -> `clean_telemetry` (validated) -> `gold_*` tables (analytics-ready)

## Step 1: Setup

In [ ]:
import os, sqlite3, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score

os.makedirs("outputs", exist_ok=True)
os.makedirs("exports", exist_ok=True)
DB = "iot_platform.db"
if os.path.exists(DB): os.remove(DB)   # fresh run
conn = sqlite3.connect(DB)
print("Setup complete")

## Step 2: Simulate device telemetry
30 devices, hourly readings for 30 days. Three devices (3, 11, 24) develop a **failure pattern** in the last 5 days
(rising vibration and temperature). The true labels are stored separately so we can later test how well the model finds them.
Then we deliberately corrupt the data (missing values, impossible values, duplicates), as real field data would be.

In [ ]:
rng = np.random.default_rng(42)
N_DEV, HOURS = 30, 24*30
ts = pd.date_range("2026-09-01", periods=HOURS, freq="h")
t = np.arange(HOURS)
FAILING = {3, 11, 24}

frames, truth = [], []
for d in range(N_DEV):
    temp = rng.normal(40, 5) + 5*np.sin(t*2*np.pi/24) + rng.normal(0, 1, HOURS)
    vib = np.abs(rng.normal(0.5, 0.1, HOURS))
    is_anom = np.zeros(HOURS, dtype=int)
    if d in FAILING:
        start = HOURS - 120
        ramp = np.clip((t - start)/120, 0, 1)
        vib = vib + ramp*rng.uniform(0.8, 1.2)
        temp = temp + ramp*rng.uniform(8, 12)
        is_anom[start+60:] = 1
    dev = f"DEV{d:03d}"
    frames.append(pd.DataFrame({
        "device_id": dev, "timestamp": ts,
        "temperature_c": temp, "vibration_g": vib,
        "battery_pct": np.linspace(100, rng.uniform(5, 60), HOURS),
        "rssi_dbm": rng.normal(-70, 5, HOURS)}))
    truth.append(pd.DataFrame({"device_id": dev, "timestamp": ts, "is_true_anomaly": is_anom}))

df = pd.concat(frames, ignore_index=True)
truth = pd.concat(truth, ignore_index=True)

# inject data-quality faults
i = rng.choice(len(df), 300, replace=False)
df.loc[i[:200], "temperature_c"] = np.nan
df.loc[i[200:260], "temperature_c"] = 999
df.loc[i[260:300], "battery_pct"] = -5
df = pd.concat([df, df.sample(300, random_state=1)])   # duplicates

df.to_csv("raw_telemetry.csv", index=False)
truth.to_sql("sim_ground_truth", conn, index=False)    # simulation only, used for model evaluation
print(df.shape)
df.head()

## Step 3: Load the RAW layer
Raw data is stored exactly as received, plus a load timestamp. It is **never edited**: all fixes go to a separate clean table, which preserves an audit trail.

In [ ]:
raw = pd.read_csv("raw_telemetry.csv")
raw["loaded_at"] = pd.Timestamp.now().isoformat()
raw.to_sql("raw_telemetry", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT COUNT(*) AS total_rows FROM raw_telemetry", conn))
pd.read_sql("""
SELECT COUNT(*) AS total_rows,
  SUM(CASE WHEN temperature_c IS NULL THEN 1 ELSE 0 END) AS missing_temp,
  SUM(CASE WHEN temperature_c > 100 THEN 1 ELSE 0 END)   AS impossible_temp,
  SUM(CASE WHEN battery_pct < 0 THEN 1 ELSE 0 END)       AS invalid_battery
FROM raw_telemetry
""", conn)

## Step 4: Clean into the CLEAN layer
Rules: remove duplicates (same device + timestamp); set impossible values to missing; interpolate missing values per device **and flag every imputed value**;
log every action in `data_quality_log`.

In [ ]:
df = pd.read_sql("SELECT * FROM raw_telemetry", conn)
df["timestamp"] = pd.to_datetime(df["timestamp"])
log = []
def record(rule, n):
    log.append({"run_at": pd.Timestamp.now().isoformat(), "rule": rule, "rows_affected": int(n)})

n = df.duplicated(subset=["device_id", "timestamp"]).sum()
record("duplicate_device_timestamp_removed", n)
df = df.drop_duplicates(subset=["device_id", "timestamp"]).copy()

bad_t = (df.temperature_c > 100) | (df.temperature_c < -40)
record("temperature_out_of_range", bad_t.sum()); df.loc[bad_t, "temperature_c"] = np.nan
bad_b = (df.battery_pct < 0) | (df.battery_pct > 100)
record("battery_out_of_range", bad_b.sum());     df.loc[bad_b, "battery_pct"] = np.nan

df["temp_imputed"] = df["temperature_c"].isna().astype(int)
df["battery_imputed"] = df["battery_pct"].isna().astype(int)
record("temperature_values_imputed", df["temp_imputed"].sum())
record("battery_values_imputed", df["battery_imputed"].sum())

df = df.sort_values(["device_id", "timestamp"])
for col in ["temperature_c", "battery_pct"]:
    df[col] = df.groupby("device_id")[col].transform(lambda s: s.interpolate(limit_direction="both"))

df.to_sql("clean_telemetry", conn, if_exists="replace", index=False)
pd.DataFrame(log).to_sql("data_quality_log", conn, if_exists="replace", index=False)
pd.read_sql("SELECT * FROM data_quality_log", conn)

## Step 5: Automated data quality checks and device quality scores
Reusable pass/fail tests on the clean layer, saved to `data_quality_checks`. Each device also gets a **quality score** (% of its raw rows with no issue),
which highlights hardware or connectivity problems.

In [ ]:
c = pd.read_sql("SELECT * FROM clean_telemetry", conn)
checks = [
    ("no_duplicate_device_timestamp", c.duplicated(["device_id","timestamp"]).sum() == 0),
    ("no_null_temperature",           c.temperature_c.isna().sum() == 0),
    ("no_null_battery",               c.battery_pct.isna().sum() == 0),
    ("temperature_in_range_-40_100",  c.temperature_c.between(-40, 100).all()),
    ("battery_in_range_0_100",        c.battery_pct.between(0, 100).all()),
    ("vibration_non_negative",        (c.vibration_g >= 0).all()),
    ("expected_row_count_21600",      len(c) == 30*24*30),
]
chk = pd.DataFrame(checks, columns=["check", "passed"])
chk["status"] = np.where(chk.passed, "PASS", "FAIL")
chk["run_at"] = pd.Timestamp.now().isoformat()
chk.drop(columns="passed").to_sql("data_quality_checks", conn, if_exists="replace", index=False)
print(chk[["check","status"]])

r = pd.read_sql("SELECT * FROM raw_telemetry", conn)
r["issue"] = (r.temperature_c.isna() | (r.temperature_c > 100) | (r.battery_pct < 0)
              | r.duplicated(["device_id","timestamp"])).astype(int)
score = (100 * (1 - r.groupby("device_id")["issue"].mean())).round(2).rename("quality_score").reset_index()
score.to_sql("gold_device_quality", conn, if_exists="replace", index=False)
score.sort_values("quality_score").head()

## Step 6: Governance documentation (data dictionary and lineage)
Stored in the database so the metadata travels with the data.

In [ ]:
dictionary = pd.DataFrame([
 ("clean_telemetry","device_id","TEXT","Unique device identifier (DEVxxx)","Device registry","Data Engineer"),
 ("clean_telemetry","timestamp","DATETIME","Hourly reading time (UTC)","Device firmware","Data Engineer"),
 ("clean_telemetry","temperature_c","REAL","Temperature in degrees Celsius. Valid range -40 to 100","Device sensor","Data Steward"),
 ("clean_telemetry","vibration_g","REAL","Vibration in g-force. Must be >= 0","Device sensor","Data Steward"),
 ("clean_telemetry","battery_pct","REAL","Battery level 0 to 100","Device firmware","Data Steward"),
 ("clean_telemetry","rssi_dbm","REAL","Wireless signal strength in dBm","Device radio","Data Steward"),
 ("clean_telemetry","temp_imputed","INTEGER","1 if temperature was estimated by interpolation","Pipeline","Data Engineer"),
 ("clean_telemetry","battery_imputed","INTEGER","1 if battery was estimated by interpolation","Pipeline","Data Engineer"),
 ("raw_telemetry","loaded_at","TEXT","Ingestion timestamp for audit trail","Pipeline","Data Engineer"),
 ("gold_device_quality","quality_score","REAL","% of raw rows with no quality issue","Pipeline","Data Steward"),
], columns=["table_name","column_name","data_type","description","source","owner_role"])
dictionary.to_sql("data_dictionary", conn, if_exists="replace", index=False)

lineage = pd.DataFrame([
 ("devices (simulated)","raw_telemetry","Ingest as-is + loaded_at"),
 ("raw_telemetry","clean_telemetry","Dedupe, range validation, flagged imputation"),
 ("raw_telemetry","gold_device_quality","Quality scoring"),
 ("clean_telemetry","gold_device_daily","Daily aggregation (SQL)"),
 ("clean_telemetry","gold_anomalies","Isolation Forest"),
 ("clean_telemetry","gold_battery_forecast","Linear trend forecast"),
], columns=["source","target","transformation"])
lineage.to_sql("data_lineage", conn, if_exists="replace", index=False)
dictionary.to_csv("exports/data_dictionary.csv", index=False)
lineage.to_csv("exports/data_lineage.csv", index=False)
lineage

## Step 7: Analytics in SQL (GOLD layer)

In [ ]:
conn.execute("DROP TABLE IF EXISTS gold_device_daily")
conn.execute("""
CREATE TABLE gold_device_daily AS
SELECT device_id, DATE(timestamp) AS day,
       ROUND(AVG(temperature_c),2) AS avg_temp, ROUND(MAX(temperature_c),2) AS max_temp,
       ROUND(AVG(vibration_g),3)   AS avg_vib,  ROUND(MAX(vibration_g),3)   AS max_vib,
       ROUND(MIN(battery_pct),1)   AS min_battery, ROUND(AVG(rssi_dbm),1) AS avg_rssi,
       COUNT(*) AS readings
FROM clean_telemetry
GROUP BY device_id, DATE(timestamp)
""")
conn.commit()

# Which devices run hottest / vibrate most in the final 5 days?
pd.read_sql("""
SELECT device_id, ROUND(AVG(avg_temp),1) AS avg_temp, ROUND(AVG(avg_vib),3) AS avg_vib, MIN(min_battery) AS min_battery
FROM gold_device_daily
WHERE day >= '2026-09-26'
GROUP BY device_id
ORDER BY avg_vib DESC
LIMIT 8
""", conn)

## Step 8: Anomaly detection (Isolation Forest)
Unsupervised model, no labels needed. Features are each reading's deviation from that device's normal behaviour (so a device that normally runs warm is not flagged unfairly).
Because this is a simulation we hold the true labels and can **measure precision and recall**.

In [ ]:
c = pd.read_sql("SELECT * FROM clean_telemetry", conn, parse_dates=["timestamp"])
c["temp_dev"] = c.temperature_c - c.groupby("device_id").temperature_c.transform("median")
c["vib_dev"]  = c.vibration_g  - c.groupby("device_id").vibration_g.transform("median")
c["vib_roll6"]  = c.groupby("device_id").vib_dev.transform(lambda s: s.rolling(6, min_periods=1).mean())
c["temp_roll6"] = c.groupby("device_id").temp_dev.transform(lambda s: s.rolling(6, min_periods=1).mean())

features = ["temp_dev", "vib_dev", "vib_roll6", "temp_roll6"]
model = IsolationForest(n_estimators=200, contamination=0.01, random_state=42)
c["anomaly"] = (model.fit_predict(c[features]) == -1).astype(int)
c["anomaly_score"] = -model.score_samples(c[features])

gt = pd.read_sql("SELECT * FROM sim_ground_truth", conn, parse_dates=["timestamp"])
ev = c.merge(gt, on=["device_id", "timestamp"])
print("Precision:", round(precision_score(ev.is_true_anomaly, ev.anomaly), 3))
print("Recall:   ", round(recall_score(ev.is_true_anomaly, ev.anomaly), 3))

flagged = c[c.anomaly == 1][["device_id","timestamp","temperature_c","vibration_g","anomaly_score"]]
flagged.to_sql("gold_anomalies", conn, if_exists="replace", index=False)
print("\nFlagged readings per device (top 6):")
print(flagged.device_id.value_counts().head(6))

## Step 9: Battery replacement forecast (predictive maintenance)
A per-device linear trend of battery level is projected forward to estimate **days until 20%**, so maintenance can be scheduled before devices go offline.

In [ ]:
rows = []
for dev, g in c.groupby("device_id"):
    hrs = np.arange(len(g))
    slope, intercept = np.polyfit(hrs, g.battery_pct.values, 1)
    now = g.battery_pct.iloc[-1]
    days_to_20 = max((now - 20) / (-slope) / 24, 0) if slope < 0 else np.nan
    rows.append({"device_id": dev, "battery_now": round(now,1),
                 "drain_pct_per_day": round(-slope*24, 2), "days_until_20pct": round(days_to_20,1)})
fc = pd.DataFrame(rows)
fc["action"] = np.select(
    [fc.battery_now <= 20, fc.days_until_20pct <= 14],
    ["REPLACE NOW (below 20%)", "REPLACE WITHIN 14 DAYS"], default="OK")
fc.to_sql("gold_battery_forecast", conn, if_exists="replace", index=False)
fc.sort_values("battery_now").head(8)

## Step 10: Charts for the README and exports for Tableau Public

In [ ]:
# 1. Failing device vs healthy device
fig, ax = plt.subplots(figsize=(10,4))
for dev, col in [("DEV003","tab:red"), ("DEV000","tab:blue")]:
    g = c[c.device_id == dev]
    ax.plot(g.timestamp, g.vibration_g, color=col, alpha=.6, label=dev)
    a = g[g.anomaly == 1]; ax.scatter(a.timestamp, a.vibration_g, color="black", s=12, zorder=3)
ax.set_title("Vibration: failing device (DEV003) vs healthy device, anomalies in black")
ax.legend(); plt.tight_layout(); plt.savefig("outputs/vibration_anomalies.png", dpi=150); plt.show()

# 2. Device quality scores
s = score.sort_values("quality_score")
fig, ax = plt.subplots(figsize=(10,4)); ax.bar(s.device_id, s.quality_score)
ax.set_ylim(90,100); ax.set_title("Data quality score by device (%)"); plt.xticks(rotation=90, fontsize=7)
plt.tight_layout(); plt.savefig("outputs/quality_scores.png", dpi=150); plt.show()

# 3. Battery forecast
f = fc.sort_values("battery_now").head(12)
fig, ax = plt.subplots(figsize=(8,4)); ax.barh(f.device_id, f.battery_now, color="tab:orange")
ax.axvline(20, color="red", ls="--"); ax.set_title("Current battery % (12 lowest devices, red line = 20% threshold)"); ax.invert_yaxis()
plt.tight_layout(); plt.savefig("outputs/battery_forecast.png", dpi=150); plt.show()

# CSV exports (Tableau Public cannot read SQLite, but reads CSV)
for tbl in ["gold_device_daily","gold_device_quality","gold_anomalies","gold_battery_forecast","data_quality_log","data_quality_checks"]:
    pd.read_sql(f"SELECT * FROM {tbl}", conn).to_csv(f"exports/{tbl}.csv", index=False)
print(os.listdir("exports"))

## Step 11: Package and download everything

In [ ]:
conn.close()
with zipfile.ZipFile("iot_project_files.zip", "w") as z:
    for folder in ["outputs", "exports"]:
        for f in os.listdir(folder): z.write(os.path.join(folder, f))
    z.write(DB); z.write("raw_telemetry.csv")
try:
    from google.colab import files
    files.download("iot_project_files.zip")
except ImportError:
    print("Not in Colab: zip saved locally")